# Bai tap: ETHUSDs (Forex)/ ETHUSDT 1m tren Forex hoac Binace
### Mot chien luoc mua: Khi gia du doan > Gia thuc te va MA 10 > MA 20
### Mot chien luoc ban: Khi gia du doan < Gia thuc te va MA 10 < MA 20

In [ ]:
# Den 21h25

# Buoc 1: Load data

In [1]:
import sys
sys.path.append('../Common')

import CommonBinanceDWH

symbol = 'ETHUSDT'
from_date = '2024-11-04'
to_date = '2024-12-07' # yyyy-mm-dd
interval = '1m'

data = CommonBinanceDWH.CommonBinanceDWH.loaddataBinance_FromTo_Split(symbol, from_date, to_date, interval)

data

No data fetched for ETHUSDT at timestamp 1733490900001


,Datetime,Open,High,Low,Close,Volume
0,2024-11-04 00:00:00,2457.73,2457.73,2455.54,2455.54,94.8930
1,2024-11-04 00:01:00,2455.54,2455.90,2454.27,2455.90,113.1605
2,2024-11-04 00:02:00,2455.90,2456.70,2453.40,2454.08,159.3716
3,2024-11-04 00:03:00,2454.08,2454.40,2453.42,2453.93,244.1911
4,2024-11-04 00:04:00,2453.92,2454.43,2453.42,2454.20,45.5168
...,...,...,...,...,...,...
46871,2024-12-06 13:11:00,3851.87,3851.87,3846.25,3848.64,431.3314
46872,2024-12-06 13:12:00,3848.63,3850.76,3847.00,3850.75,385.8811
46873,2024-12-06 13:13:00,3850.75,3852.75,3849.00,3850.57,265.9881
46874,2024-12-06 13:14:00,3850.57,3854.77,3850.57,3854.21,309.4967


In [2]:
data.to_csv('data.csv')

# Buoc 2: Dua vao data:
### Tinh ra chi bao MA10, MA20
### Du doan dua va Hoi quy tuyen tinh
### Tinh Buy_Signal, Sell_Signal
### Day qua Redis

In [ ]:
import talib
from sklearn.linear_model import LinearRegression 
import redis 
from datetime import datetime, timedelta
import time
import pandas as pd 
import numpy as np

data['MA10'] = talib.SMA(data['Close'], timeperiod=10)
data['MA20'] = talib.SMA(data['Close'], timeperiod=20)

# Xu ly NA: Drop NA
data = data.dropna()

# Khoi tao Feature va Target
X = data[['Open', 'High', 'Low', 'Volume', 'MA10', 'MA20']] # Feature
y = data['Close'] # Target

model = LinearRegression()
model.fit(X, y)

# Du doan
data['Predicted_Close'] = model.predict(X)

# Tinh toan Buy_Signal va Sell Signal
data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['MA10'] > data['MA20']))
data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['MA10'] < data['MA20']))

# Day sang Redis
# Tạo kết nối Redis
r = redis.Redis(host='localhost', port=6379, db=15) # DB muon chua # CK 0-5; FX 6-10; Crypto 11-15
# Tạo hash key
hash_key = 'OG_ML_CRTO_MA10, MA20'

last_record = data.iloc[-1] # Lay record moi nhat

# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
	for field, value in last_record.to_dict().items():
		# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
		if isinstance(value, pd.Timestamp):
			value = value.isoformat()
		elif isinstance(value, (int, np.uint64)):
			value = str(value)
		r.hset(hash_key, field, value)  
		r.hset(hash_key, 'Symbol', symbol)
		r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
		last_record['Insertdate'] = datetime.now().isoformat()
	print(last_record)   
else:
	print(last_record)   
	print('Không có tín hiệu!')

Datetime                  2024-12-04 15:00:00
Open                                   3778.9
High                                  3779.99
Low                                   3763.84
Close                                 3774.02
Volume                              2596.9611
MA10                                 3782.286
MA20                                3768.8865
Predicted_Close                   3769.307756
Buy_Signal                               True
Sell_Signal                             False
Insertdate         2024-12-04T22:00:49.350997
Name: 900, dtype: object


C:\Users\dongn\AppData\Local\Temp\ipykernel_13308\496643273.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted_Close'] = model.predict(X)
C:\Users\dongn\AppData\Local\Temp\ipykernel_13308\496643273.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['MA10'] > data['MA20']))
C:\Users\dongn\AppData\Local\Temp\ipykernel_13308\496643273.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

In [23]:
data

,Datetime,Open,High,Low,Close,Volume,MA10,MA20,Predicted_Close,Buy_Signal,Sell_Signal
38,2024-12-04 00:38:00,3635.92,3635.92,3626.10,3627.38,510.8805,3635.882,3631.3835,3628.814721,True,False
39,2024-12-04 00:39:00,3627.37,3627.42,3622.61,3622.62,343.8103,3634.925,3631.1835,3623.779887,True,False
40,2024-12-04 00:40:00,3622.62,3623.13,3619.98,3622.37,819.3456,3634.031,3631.0770,3620.853713,True,False
41,2024-12-04 00:41:00,3622.36,3630.46,3622.36,3626.74,607.4896,3633.184,3631.1775,3629.587960,True,False
42,2024-12-04 00:42:00,3626.74,3628.74,3625.25,3628.00,412.1802,3631.748,3631.4255,3627.241505,True,False
...,...,...,...,...,...,...,...,...,...,...,...
891,2024-12-04 14:51:00,3766.57,3766.57,3759.21,3761.99,1715.2323,3756.925,3746.9910,3760.945018,True,False
892,2024-12-04 14:52:00,3762.00,3773.29,3760.29,3771.47,1800.4456,3759.182,3749.3265,3770.860362,True,False
893,2024-12-04 14:53:00,3771.48,3780.00,3770.80,3780.00,1293.5571,3762.844,3751.7470,3778.468740,True,False
894,2024-12-04 14:54:00,3779.99,3792.78,3776.16,3792.37,5543.0388,3767.401,3754.5855,3788.303645,True,False


In [17]:
data.to_csv('data.csv')

# Du doan tuong lai

In [ ]:
import talib
from sklearn.linear_model import LinearRegression 
import redis 
from datetime import datetime, timedelta
import time
import pandas as pd 
import numpy as np

import talib
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np

# Tính MA10 và MA20 sử dụng talib
data['MA10'] = talib.SMA(data['Close'].values, timeperiod=10)
data['MA20'] = talib.SMA(data['Close'].values, timeperiod=20)

# Loại bỏ giá trị NaN do MA10 và MA20 sinh ra
data = data.dropna()

# Chuẩn bị các đặc trưng cho từng mô hình
features_close = ['Open', 'High', 'Low', 'Volume', 'MA10', 'MA20']
features_open = ['High', 'Low', 'Close', 'Volume', 'MA10', 'MA20']
features_high = ['Open', 'Low', 'Close', 'Volume', 'MA10', 'MA20']
features_low = ['Open', 'High', 'Close', 'Volume', 'MA10', 'MA20']
features_volume = ['Open', 'High', 'Low', 'Close', 'MA10', 'MA20']

# Chuẩn bị dữ liệu
X_close = data[features_close]
y_close = data['Close']

X_open = data[features_open]
y_open = data['Open']

X_high = data[features_high]
y_high = data['High']

X_low = data[features_low]
y_low = data['Low']

X_volume = data[features_volume]
y_volume = data['Volume']

# Huấn luyện mô hình dự đoán Close, Open, High, Low, Volume
model_close = LinearRegression().fit(X_close, y_close)
model_open = LinearRegression().fit(X_open, y_open)
model_high = LinearRegression().fit(X_high, y_high)
model_low = LinearRegression().fit(X_low, y_low)
model_volume = LinearRegression().fit(X_volume, y_volume)

# Dự đoán giá trị Close cho toàn bộ dữ liệu hiện tại
data['Predicted_Close'] = model_close.predict(X_close)

# Dự đoán giá trị ngày mai
future_features = pd.DataFrame({
    'Open': [model_open.predict(X_open.iloc[-1:])[0]],
    'High': [model_high.predict(X_high.iloc[-1:])[0]],
    'Low': [model_low.predict(X_low.iloc[-1:])[0]],
    'Volume': [model_volume.predict(X_volume.iloc[-1:])[0]],
    'MA10': [pd.concat([data['Close'].iloc[-9:], pd.Series([model_close.predict(X_close.iloc[-1:])[0]])], ignore_index=True).mean()],
    'MA20': [pd.concat([data['Close'].iloc[-19:], pd.Series([model_close.predict(X_close.iloc[-1:])[0]])], ignore_index=True).mean() if len(data['Close']) >= 19 else data['Close'].mean()]
})

# Sử dụng các đặc trưng dự đoán để tính giá Close của ngày mai
future_features['Close'] = model_close.predict(future_features)

# Tạo DataFrame riêng `dataTomorow` chứa dự đoán cho ngày mai
dataTomorow = pd.DataFrame({
    'Datetime': [data['Datetime'].iloc[-1] + pd.Timedelta(days=1)],
    'Open': future_features['Open'],
    'High': future_features['High'],
    'Low': future_features['Low'],
    'Close': future_features['Close'],
    'Volume': future_features['Volume'],
    'MA10': future_features['MA10'],
    'MA20': future_features['MA20'],
    'Predicted_Close': future_features['Close'],  # Dự đoán cho ngày mai
    'Predicted_Tomorrow_Close': future_features['Close']  # Thêm cột này
})

# Dự đoán giá Close cho ngày mai trong DataFrame gốc
data['Predicted_Tomorrow_Close'] = np.nan
if not future_features.empty:
    predicted_tomorrow_close = future_features['Close'].iloc[0]
    data.at[data.index[-1], 'Predicted_Tomorrow_Close'] = predicted_tomorrow_close

# Xác định tín hiệu giao dịch
data['Buy_Signal'] = (data['Predicted_Close'] > data['Close']) & (data['MA10'] > data['MA20'])
data['Sell_Signal'] = (data['Predicted_Close'] < data['Close']) & (data['MA10'] < data['MA20'])

# In kết quả
data

C:\Users\PCDTT\AppData\Local\Temp\ipykernel_17228\1832485525.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted_Close'] = model_close.predict(X_close)
C:\Users\PCDTT\AppData\Local\Temp\ipykernel_17228\1832485525.py:82: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted_Tomorrow_Close'] = np.nan
C:\Users\PCDTT\AppData\Local\Temp\ipykernel_17228\1832485525.py:88: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,

,Datetime,Open,High,Low,Close,Volume,MA10,MA20,Predicted_Close,Predicted_Tomorrow_Close,Buy_Signal,Sell_Signal
19,2024-12-04 00:19:00,3628.58,3628.87,3622.01,3626.62,372.4924,3625.303,3622.5575,3624.087201,NaN,False,False
20,2024-12-04 00:20:00,3626.62,3629.00,3624.50,3624.50,243.8051,3625.827,3622.8040,3627.014650,NaN,True,False
21,2024-12-04 00:21:00,3624.50,3625.44,3623.28,3624.73,160.1078,3626.001,3623.0715,3624.288592,NaN,False,False
22,2024-12-04 00:22:00,3624.73,3624.73,3618.53,3623.04,408.8690,3625.846,3623.3140,3620.297167,NaN,False,False
23,2024-12-04 00:23:00,3623.04,3624.32,3620.81,3621.30,210.6849,3625.096,3623.4105,3622.438008,NaN,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
3623,2024-12-06 12:23:00,3878.62,3878.62,3872.17,3872.17,282.2644,3876.840,3877.6145,3873.799091,NaN,False,False
3624,2024-12-06 12:24:00,3872.18,3874.00,3871.40,3873.66,198.9687,3876.187,3877.5185,3872.889133,NaN,False,True
3625,2024-12-06 12:25:00,3873.65,3876.32,3873.65,3875.40,250.1687,3875.867,3877.6385,3875.605545,NaN,False,False
3626,2024-12-06 12:26:00,3875.40,3876.77,3874.60,3875.38,142.5814,3875.646,3877.6670,3875.650852,NaN,False,False


In [53]:
dataTomorow

,Datetime,Open,High,Low,Close,Volume,MA10,MA20,Predicted_Close,Predicted_Tomorrow_Close
0,2024-12-07 12:27:00,3875.137564,3875.973007,3873.269609,3874.188678,146.703671,3874.988934,3877.278967,3874.188678,3874.188678
